# Monte Carlo Scenario Analysis

This notebook analyzes the augmented inference outputs from the MC-MoE model's scenario generation pipeline.

The scenario generation works by mutating individual fundamental feature channels (e.g., DuPont net margin, leverage, profitability) with historical quantile values, then running inference to see how the model's price forecasts change under different fundamental regimes.

**Analyses:**
1. Fan charts per ticker (prediction percentile bands across scenarios)
2. Feature sensitivity analysis (which fundamentals most affect forecasts)
3. Empirical confidence intervals at each forecast horizon
4. Scenario-based Value-at-Risk (VaR) and Conditional VaR (CVaR)

In [ ]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import os
import sys
import logging

# Setup
sns.set_theme(style='whitegrid', font_scale=1.1)
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')

# Add project paths
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from ML_Core.src.utils.utils import load_config, config_to_args

# Load config to get version name and paths
config_file_path = '../config/config_tickers_448_7_Channels_with_temporal_tape_v2.yaml'
config_dict = load_config(config_file_path)
place_holders_to_replace = {'version_name': config_dict['regression']['common']['version_name']}
args = config_to_args(config_dict, place_holders_to_replace)

VERSION_NAME = args.regression.common.version_name
print(f'Version: {VERSION_NAME}')

## 1. Load and Parse Scenario Data

In [ ]:
# Load the pickle output from time_moe_features_inferencing.ipynb
seq_path = args.regression.S6.output_seq_path
logging.info(f'Loading augmented inference sequences from: {seq_path}')

with open(seq_path, 'rb') as f:
    all_batches = pickle.load(f)

logging.info(f'Loaded {len(all_batches)} batches')

# Flatten batches into a DataFrame
rows = []
for batch in all_batches:
    B = len(batch['ticker'])
    for i in range(B):
        forecast = batch['model_prediction_sequence'][i]
        if hasattr(forecast, 'numpy'):
            forecast = forecast.numpy()
        
        rows.append({
            'ticker': batch['ticker'][i],
            'augmented_feature': batch['augmented_feature'][i],
            'date': batch['date_seq'][i][-1],  # last date in context window = prediction origin
            'forecast': forecast,  # [8] predicted prices for day+1 through day+8
        })

df = pd.DataFrame(rows)
df['date'] = pd.to_datetime(df['date'])
df['is_original'] = df['augmented_feature'] == 'none'

print(f'Total samples: {len(df):,}')
print(f'Original (unperturbed) samples: {df["is_original"].sum():,}')
print(f'Augmented (scenario) samples: {(~df["is_original"]).sum():,}')
print(f'Unique tickers: {df["ticker"].nunique()}')
print(f'Unique dates: {df["date"].nunique()}')
print(f'Date range: {df["date"].min()} to {df["date"].max()}')
print(f'Features mutated: {df[~df["is_original"]]["augmented_feature"].unique()}')
print(f'Forecast shape per sample: {df.iloc[0]["forecast"].shape}')

In [ ]:
# Split into original and augmented DataFrames
df_original = df[df['is_original']].copy()
df_augmented = df[~df['is_original']].copy()

# Verify augmentation factor
n_features = df_augmented['augmented_feature'].nunique()
expected_augmentation = len(df_original) * n_features * 5  # 5 quantile points per feature
print(f'Number of features mutated: {n_features}')
print(f'Expected augmented samples (original x features x quantiles): {expected_augmentation:,}')
print(f'Actual augmented samples: {len(df_augmented):,}')

# Samples per ticker-date
samples_per_group = df.groupby(['ticker', 'date']).size()
print(f'\nSamples per (ticker, date): min={samples_per_group.min()}, max={samples_per_group.max()}, mean={samples_per_group.mean():.1f}')

## 2. Prediction Fan Charts

For selected tickers, visualize how the 8-step price forecast varies across scenarios.
The fan chart shows percentile bands (5th/25th/50th/75th/95th) across all scenario forecasts,
with the original (unperturbed) forecast overlaid in red.

In [ ]:
def plot_fan_charts(ticker, df, num_dates=5, figsize=(20, 4)):
    """
    Plot prediction fan charts for a ticker across evenly-spaced sample dates.
    
    Each subplot shows the 8-step forecast distribution for one prediction origin date.
    Blue bands = scenario percentiles, red dashed = original (unperturbed) forecast.
    """
    ticker_data = df[df['ticker'] == ticker]
    dates = sorted(ticker_data['date'].unique())
    
    if len(dates) == 0:
        print(f'No data for ticker {ticker}')
        return
    
    # Evenly sample dates
    step = max(1, len(dates) // num_dates)
    sample_dates = dates[::step][:num_dates]
    
    fig, axes = plt.subplots(1, len(sample_dates), figsize=figsize, sharey=True)
    if len(sample_dates) == 1:
        axes = [axes]
    
    horizons = np.arange(1, 9)
    
    for ax, date in zip(axes, sample_dates):
        scenarios = ticker_data[ticker_data['date'] == date]
        forecasts = np.stack(scenarios['forecast'].values)  # [N_scenarios, 8]
        
        p5  = np.percentile(forecasts, 5, axis=0)
        p25 = np.percentile(forecasts, 25, axis=0)
        p50 = np.percentile(forecasts, 50, axis=0)
        p75 = np.percentile(forecasts, 75, axis=0)
        p95 = np.percentile(forecasts, 95, axis=0)
        
        ax.fill_between(horizons, p5, p95, alpha=0.15, color='steelblue', label='5-95%')
        ax.fill_between(horizons, p25, p75, alpha=0.30, color='steelblue', label='25-75%')
        ax.plot(horizons, p50, 'b-', linewidth=2, label='Median')
        
        # Overlay original forecast
        orig = scenarios[scenarios['is_original']]
        if len(orig) > 0:
            ax.plot(horizons, orig.iloc[0]['forecast'], 'r--', linewidth=1.5, label='Original')
        
        ax.set_title(pd.Timestamp(date).strftime('%Y-%m-%d'), fontsize=10)
        ax.set_xlabel('Horizon (days)')
        ax.set_xticks(horizons)
    
    axes[0].set_ylabel('Predicted Price')
    axes[-1].legend(loc='upper right', fontsize=8)
    fig.suptitle(f'Prediction Fan Charts: {ticker}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    return fig

In [ ]:
# Select representative tickers to visualize
# Pick tickers with the most scenario data available
ticker_counts = df.groupby('ticker').size().sort_values(ascending=False)
top_tickers = ticker_counts.head(4).index.tolist()

print(f'Plotting fan charts for: {top_tickers}')

# Create output directory
figures_dir = '../data/outputs/figures'
os.makedirs(figures_dir, exist_ok=True)

for ticker in top_tickers:
    fig = plot_fan_charts(ticker, df, num_dates=5)
    if fig:
        fig.savefig(f'{figures_dir}/fan_chart_{ticker}.png', dpi=150, bbox_inches='tight')
        plt.show()

## 3. Feature Sensitivity Analysis

Quantify which fundamental features cause the most forecast dispersion when mutated.
Higher sensitivity = the model's price forecast is more responsive to changes in that feature channel.

In [ ]:
def compute_feature_sensitivity(df_augmented, df_original):
    """
    For each mutated feature, compute the mean absolute deviation of the augmented
    forecast from the original (unperturbed) baseline forecast.
    
    Returns a DataFrame ranked by forecast impact.
    """
    # Build a lookup: (ticker, date) -> original forecast
    baseline_lookup = {}
    for _, row in df_original.iterrows():
        key = (row['ticker'], row['date'])
        baseline_lookup[key] = row['forecast']
    
    # Compute deviations
    results = []
    for _, row in df_augmented.iterrows():
        key = (row['ticker'], row['date'])
        baseline = baseline_lookup.get(key)
        if baseline is None:
            continue
        
        deviation = np.abs(row['forecast'] - baseline)
        # Also compute relative deviation (percentage)
        rel_deviation = deviation / (np.abs(baseline) + 1e-8) * 100
        
        results.append({
            'feature': row['augmented_feature'],
            'mean_abs_deviation': deviation.mean(),
            'max_abs_deviation': deviation.max(),
            'mean_rel_deviation_pct': rel_deviation.mean(),
        })
    
    sensitivity_df = pd.DataFrame(results)
    summary = sensitivity_df.groupby('feature').agg({
        'mean_abs_deviation': ['mean', 'std'],
        'max_abs_deviation': 'mean',
        'mean_rel_deviation_pct': 'mean',
    }).round(4)
    
    # Flatten column names
    summary.columns = ['mean_abs_dev', 'std_abs_dev', 'mean_max_dev', 'mean_rel_dev_pct']
    summary = summary.sort_values('mean_abs_dev', ascending=False)
    
    return summary


logging.info('Computing feature sensitivity...')
sensitivity = compute_feature_sensitivity(df_augmented, df_original)
print('\nFeature Sensitivity Ranking (by mean absolute forecast deviation):')
print('=' * 80)
print(sensitivity.to_string())

In [ ]:
# Plot feature sensitivity bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Absolute deviation
ax = axes[0]
sensitivity['mean_abs_dev'].plot(kind='barh', ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('Mean Absolute Forecast Deviation')
ax.set_title('Feature Sensitivity (Absolute)')
ax.invert_yaxis()

# Relative deviation
ax = axes[1]
sensitivity['mean_rel_dev_pct'].plot(kind='barh', ax=ax, color='coral', edgecolor='black')
ax.set_xlabel('Mean Relative Forecast Deviation (%)')
ax.set_title('Feature Sensitivity (Relative %)')
ax.invert_yaxis()

fig.suptitle('Which Fundamental Features Most Affect Price Forecasts?', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{figures_dir}/feature_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Empirical Confidence Intervals by Horizon

Compute the 90% confidence interval width at each forecast horizon (day+1 through day+8),
averaged across all ticker-date combinations. We expect CI width to increase with horizon.

In [ ]:
def confidence_intervals_by_horizon(df, ci_level=90):
    """
    Compute CI width at each forecast horizon, averaged across all ticker-date groups.
    
    Args:
        df: DataFrame with all samples (original + augmented)
        ci_level: Confidence level (default 90%)
    
    Returns:
        DataFrame with columns: horizon, mean_ci_width, std_ci_width, mean_ci_width_pct
    """
    lower_pct = (100 - ci_level) / 2
    upper_pct = 100 - lower_pct
    
    ci_data = {h: [] for h in range(8)}
    ci_data_pct = {h: [] for h in range(8)}
    
    for (ticker, date), group in df.groupby(['ticker', 'date']):
        forecasts = np.stack(group['forecast'].values)  # [N, 8]
        if forecasts.shape[0] < 3:  # Need enough scenarios
            continue
        
        for h in range(8):
            vals = forecasts[:, h]
            ci_width = np.percentile(vals, upper_pct) - np.percentile(vals, lower_pct)
            ci_data[h].append(ci_width)
            
            # Relative CI width (as % of median)
            median_val = np.median(vals)
            if abs(median_val) > 1e-8:
                ci_data_pct[h].append(ci_width / abs(median_val) * 100)
    
    results = []
    for h in range(8):
        results.append({
            'horizon': h + 1,
            'mean_ci_width': np.mean(ci_data[h]),
            'std_ci_width': np.std(ci_data[h]),
            'mean_ci_width_pct': np.mean(ci_data_pct[h]) if ci_data_pct[h] else 0,
        })
    
    return pd.DataFrame(results)


logging.info('Computing confidence intervals by horizon...')
ci_df = confidence_intervals_by_horizon(df, ci_level=90)
print(f'\n90% Confidence Interval Width by Forecast Horizon:')
print('=' * 70)
print(ci_df.to_string(index=False))

In [ ]:
# Plot CI width vs horizon
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Absolute CI width
ax = axes[0]
ax.plot(ci_df['horizon'], ci_df['mean_ci_width'], 'o-', color='steelblue', linewidth=2, markersize=8)
ax.fill_between(ci_df['horizon'],
                ci_df['mean_ci_width'] - ci_df['std_ci_width'],
                ci_df['mean_ci_width'] + ci_df['std_ci_width'],
                alpha=0.2, color='steelblue')
ax.set_xlabel('Forecast Horizon (days)')
ax.set_ylabel('90% CI Width (price units)')
ax.set_title('Absolute CI Width')
ax.set_xticks(range(1, 9))

# Relative CI width (%)
ax = axes[1]
ax.plot(ci_df['horizon'], ci_df['mean_ci_width_pct'], 'o-', color='coral', linewidth=2, markersize=8)
ax.set_xlabel('Forecast Horizon (days)')
ax.set_ylabel('90% CI Width (% of median forecast)')
ax.set_title('Relative CI Width')
ax.set_xticks(range(1, 9))

fig.suptitle('Forecast Uncertainty Grows with Horizon', fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(f'{figures_dir}/ci_width_by_horizon.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Scenario-Based Value-at-Risk (VaR) and CVaR

Compute VaR and CVaR at each forecast horizon using the scenario return distribution.
For each ticker-date, scenario returns are computed relative to the original forecast.

In [ ]:
def compute_scenario_var(df, df_original, alpha=0.05):
    """
    Compute scenario-based VaR and CVaR at each forecast horizon.
    
    Returns are computed as: (scenario_price - original_price) / original_price
    VaR = alpha-th percentile of the return distribution (worst-case loss)
    CVaR = mean of returns below VaR (expected shortfall)
    """
    # Build original forecast lookup
    baseline_lookup = {}
    for _, row in df_original.iterrows():
        key = (row['ticker'], row['date'])
        baseline_lookup[key] = row['forecast']
    
    # Collect returns per horizon
    horizon_returns = {h: [] for h in range(8)}
    
    for _, row in df.iterrows():
        key = (row['ticker'], row['date'])
        baseline = baseline_lookup.get(key)
        if baseline is None:
            continue
        
        for h in range(8):
            if abs(baseline[h]) > 1e-8:
                ret = (row['forecast'][h] - baseline[h]) / abs(baseline[h])
                horizon_returns[h].append(ret)
    
    var_results = []
    for h in range(8):
        returns = np.array(horizon_returns[h])
        if len(returns) == 0:
            continue
        
        var_val = np.percentile(returns, alpha * 100)
        cvar_val = returns[returns <= var_val].mean() if (returns <= var_val).any() else var_val
        
        var_results.append({
            'horizon': h + 1,
            f'VaR_{int(alpha*100)}%': round(var_val * 100, 4),  # as percentage
            f'CVaR_{int(alpha*100)}%': round(cvar_val * 100, 4),
            'mean_return_%': round(returns.mean() * 100, 4),
            'std_return_%': round(returns.std() * 100, 4),
            'n_scenarios': len(returns),
        })
    
    return pd.DataFrame(var_results)


logging.info('Computing scenario-based VaR...')
var_df = compute_scenario_var(df, df_original, alpha=0.05)
print(f'\nScenario-Based VaR/CVaR by Forecast Horizon (returns in %):')
print('=' * 90)
print(var_df.to_string(index=False))

In [ ]:
# Plot VaR and CVaR by horizon
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(var_df['horizon'], var_df['VaR_5%'], 'o-', color='steelblue', linewidth=2, markersize=8, label='5% VaR')
ax.plot(var_df['horizon'], var_df['CVaR_5%'], 's--', color='firebrick', linewidth=2, markersize=8, label='5% CVaR (Expected Shortfall)')
ax.axhline(y=0, color='grey', linestyle=':', alpha=0.5)

ax.set_xlabel('Forecast Horizon (days)')
ax.set_ylabel('Return (%)')
ax.set_title('Scenario-Based VaR and CVaR by Horizon', fontsize=13, fontweight='bold')
ax.set_xticks(range(1, 9))
ax.legend()

plt.tight_layout()
fig.savefig(f'{figures_dir}/var_cvar_by_horizon.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Return Distribution per Horizon

Visualize the full scenario return distribution at selected horizons.

In [ ]:
def plot_return_distributions(df, df_original, horizons_to_plot=[1, 4, 8]):
    """
    Plot histograms of scenario returns at selected horizons.
    """
    # Build original forecast lookup
    baseline_lookup = {}
    for _, row in df_original.iterrows():
        key = (row['ticker'], row['date'])
        baseline_lookup[key] = row['forecast']
    
    fig, axes = plt.subplots(1, len(horizons_to_plot), figsize=(5 * len(horizons_to_plot), 4))
    if len(horizons_to_plot) == 1:
        axes = [axes]
    
    for ax, h in zip(axes, horizons_to_plot):
        returns = []
        for _, row in df.iterrows():
            key = (row['ticker'], row['date'])
            baseline = baseline_lookup.get(key)
            if baseline is not None and abs(baseline[h - 1]) > 1e-8:
                ret = (row['forecast'][h - 1] - baseline[h - 1]) / abs(baseline[h - 1]) * 100
                returns.append(ret)
        
        returns = np.array(returns)
        # Clip for visualization
        plot_returns = np.clip(returns, -10, 10)
        
        ax.hist(plot_returns, bins=80, density=True, alpha=0.7, color='steelblue', edgecolor='white')
        ax.axvline(x=0, color='black', linestyle='-', alpha=0.3)
        ax.axvline(x=np.percentile(returns, 5), color='red', linestyle='--', alpha=0.7, label=f'5% VaR: {np.percentile(returns, 5):.2f}%')
        ax.set_xlabel('Scenario Return (%)')
        ax.set_ylabel('Density')
        ax.set_title(f'Horizon: Day+{h}')
        ax.legend(fontsize=8)
    
    fig.suptitle('Scenario Return Distributions', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig.savefig(f'{figures_dir}/return_distributions.png', dpi=150, bbox_inches='tight')
    return fig


plot_return_distributions(df, df_original, horizons_to_plot=[1, 4, 8])
plt.show()

## 7. Summary Statistics Table

Consolidate all results into a single summary for the dissertation report.

In [ ]:
print('=' * 90)
print('SCENARIO ANALYSIS SUMMARY')
print('=' * 90)

print(f'\n--- Dataset ---')
print(f'Date range: {df["date"].min().strftime("%Y-%m-%d")} to {df["date"].max().strftime("%Y-%m-%d")}')
print(f'Tickers: {df["ticker"].nunique()}')
print(f'Trading days: {df["date"].nunique()}')
print(f'Total scenarios: {len(df):,} (original: {len(df_original):,}, augmented: {len(df_augmented):,})')
print(f'Scenarios per ticker-date: {df.groupby(["ticker", "date"]).size().mean():.0f}')

print(f'\n--- Feature Sensitivity Ranking ---')
print(sensitivity[['mean_abs_dev', 'mean_rel_dev_pct']].to_string())

print(f'\n--- 90% Confidence Interval Width by Horizon ---')
print(ci_df[['horizon', 'mean_ci_width', 'mean_ci_width_pct']].to_string(index=False))

print(f'\n--- 5% VaR / CVaR by Horizon (%) ---')
print(var_df[['horizon', 'VaR_5%', 'CVaR_5%']].to_string(index=False))

print(f'\n--- Figures saved to: {os.path.abspath(figures_dir)} ---')
for f in os.listdir(figures_dir):
    print(f'  {f}')